# Expert annotations for the CTU-UHB database

Link to expert evaluation as described in Hruban et al. 2015 - http://people.ciirc.cvut.cz/~spilkjir/data.html

Expert evaluation of the CTG data "Gold Standard" evaluation based on annotation of the signals by 9 expert obstetricians (following FIGO guidelines used in the Czech Republic) including variability/confidence for each signal

9 expert obstetricians analysed the CTG data from dataset The CTGAnnotator was used to obtain annotation of the CTG recordings from nine expert-obstetricians. The CTGAnnotator presented the CTG trace in form of consecutive 30-minute windows together with basic clinical information. Each window was expected to be evaluated based on FIGO criteria by assigning it to one of the three classes (Normal/Suspicious/Pathological). All obstetricians working on delivery wards of six Obstetrics and Gynecology Departments of all the University Hospitals in the Czech Republic have been currently practicing delivery ward doctors with median experience of 15 years (minimum 10, maximum 33). 

I used majority voting to create one label for each record as 9 obstetricions did not record the same results due to inter-observer variability and clinical subjectivity. 

In [ ]:
import pandas as pd
import numpy as np

In [17]:
file_path = "..\ExpertAnnotations\ExpertAnnCTU-UHB-CTG_20150203 (1).xls"
df = pd.read_csv(file_path)

In [18]:
# Data Cleaning and Focus 
# Focus only on the record ID and the final outcome prediction (eval_step4)
# eval_step4 values: 1=no hypoxia, 2=mild hypoxia, 3=severe hypoxia, -1=uninterpretable
df_votes = df[['rec_id', 'eval_step4']].copy()

In [19]:
# Filter out uninterpretable votes (-1). Majority Vote relies only on valid opinions.
df_filtered = df_votes[df_votes['eval_step4'] != -1]

In [20]:
# Calculate Majority Vote

# Function to calculate the mode (most frequent value)
# .mode() returns the most frequent value(s). We use [0] to select the first one 
# in case of a tie (which is a standard way to handle ties in this context).
def get_majority_vote(series):
    return series.mode()[0]

In [21]:
# Group by rec_id and apply the majority voting function
majority_labels = df_filtered.groupby('rec_id')['eval_step4'].agg(get_majority_vote).reset_index()
majority_labels.columns = ['rec_id', 'Majority_Vote_Label']

In [22]:
# Display Results and Save

# Include the -1 mapping for completeness, though it shouldn't appear in the final labels.
label_map = {
    1: 'No Hypoxia (Normal)', 
    2: 'Mild Hypoxia (Suspicious)', 
    3: 'Severe Hypoxia (Pathological)',
    -1: 'Uninterpretable (Filtered)' # This label will not appear in the final output.
}
majority_labels['Clinical_Label'] = majority_labels['Majority_Vote_Label'].map(label_map)

In [23]:
# Display the first few records
print("--- Final Majority Vote Labels for the First 10 Records ---")
print(majority_labels.head(10).to_markdown(index=False))

--- Final Majority Vote Labels for the First 10 Records ---
|   rec_id |   Majority_Vote_Label | Clinical_Label            |
|---------:|----------------------:|:--------------------------|
|     1001 |                     1 | No Hypoxia (Normal)       |
|     1002 |                     2 | Mild Hypoxia (Suspicious) |
|     1003 |                     2 | Mild Hypoxia (Suspicious) |
|     1004 |                     2 | Mild Hypoxia (Suspicious) |
|     1005 |                     1 | No Hypoxia (Normal)       |
|     1006 |                     1 | No Hypoxia (Normal)       |
|     1007 |                     2 | Mild Hypoxia (Suspicious) |
|     1008 |                     1 | No Hypoxia (Normal)       |
|     1009 |                     2 | Mild Hypoxia (Suspicious) |
|     1010 |                     2 | Mild Hypoxia (Suspicious) |


In [24]:
# Display the class distribution (confirming only 1, 2, or 3 appear)
print("\n--- Distribution of Final Labels ---")
print(majority_labels['Clinical_Label'].value_counts(normalize=True).to_markdown())


--- Distribution of Final Labels ---
|                           |   Clinical_Label |
|:--------------------------|-----------------:|
| No Hypoxia (Normal)       |         0.771739 |
| Mild Hypoxia (Suspicious) |         0.228261 |


In [26]:
# Save the final labels to a new CSV file
majority_labels.to_csv("..\ExpertAnnotations\CTG_Majority_Vote_Labels_FINAL.csv", index=False)
print("\nLabels saved to 'CTG_Majority_Vote_Labels_FINAL.csv' in ExpertAnnotations folder.")


Labels saved to 'CTG_Majority_Vote_Labels_FINAL.csv' in ExpertAnnotations folder.
